<a href="https://colab.research.google.com/github/sreedevisreekumar/pysparklearning/blob/main/pysparklearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
  import pandas as pd
  from pyspark.sql import SparkSession
except Exception as e:
  print(e)

In [ ]:
spark = SparkSession.builder\
.appName("MyProcess")\
.master("local[*]")\
.getOrCreate()

In [ ]:
spark

In [ ]:
header = ["city", "type", "price"]

In [ ]:
data = map(lambda r: (r[0], r[1], float(r[2])),map(lambda x:x.split(","),["Paris,Food,19.00","Marseille,Clothing,12.00","Paris,Food,8.00","Paris,Clothing,15.00","Marseille,Food,20.00","Lyon,Book,10.00"]))

In [ ]:
# see the raw data before it becomes a data frame
data_as_list =list(data)
print("Raw data before DataFrame : ")
for row in data_as_list:
  print(f" {row}")


Raw data before DataFrame : 
 ('Paris', 'Food', 19.0)
 ('Marseille', 'Clothing', 12.0)
 ('Paris', 'Food', 8.0)
 ('Paris', 'Clothing', 15.0)
 ('Marseille', 'Food', 20.0)
 ('Lyon', 'Book', 10.0)


In [ ]:
df= spark.createDataFrame(data_as_list,header)
df.show()

+---------+--------+-----+
|     city|    type|price|
+---------+--------+-----+
|    Paris|    Food| 19.0|
|Marseille|Clothing| 12.0|
|    Paris|    Food|  8.0|
|    Paris|Clothing| 15.0|
|Marseille|    Food| 20.0|
|     Lyon|    Book| 10.0|
+---------+--------+-----+



In [ ]:
df.take(2)

[Row(city='Paris', type='Food', price=19.0),
 Row(city='Marseille', type='Clothing', price=12.0)]

In [ ]:
df

DataFrame[city: string, type: string, price: double]

In [ ]:
df.dtypes

[('city', 'string'), ('type', 'string'), ('price', 'double')]

In [ ]:
df.explain()

== Physical Plan ==
*(1) Scan ExistingRDD[city#0,type#1,price#2]




In [ ]:
df.select("city").show()

+---------+
|     city|
+---------+
|    Paris|
|Marseille|
|    Paris|
|    Paris|
|Marseille|
|     Lyon|
+---------+



In [ ]:
df.select(["city","type"]).show()

+---------+--------+
|     city|    type|
+---------+--------+
|    Paris|    Food|
|Marseille|Clothing|
|    Paris|    Food|
|    Paris|Clothing|
|Marseille|    Food|
|     Lyon|    Book|
+---------+--------+



In [ ]:
df.select(["city","price"]).take(3)

[Row(city='Paris', price=19.0),
 Row(city='Marseille', price=12.0),
 Row(city='Paris', price=8.0)]

In [ ]:
df.collect()

[Row(city='Paris', type='Food', price=19.0),
 Row(city='Marseille', type='Clothing', price=12.0),
 Row(city='Paris', type='Food', price=8.0),
 Row(city='Paris', type='Clothing', price=15.0),
 Row(city='Marseille', type='Food', price=20.0),
 Row(city='Lyon', type='Book', price=10.0)]

In [ ]:
df.printSchema()

root
 |-- city: string (nullable = true)
 |-- type: string (nullable = true)
 |-- price: double (nullable = true)



In [ ]:
from pyspark.sql.types import StringType, FloatType, StructType, StructField

In [ ]:
data = map(lambda r: (r[0], r[1], float(r[2])),
map(lambda x: x.split(","),["Paris,Food,19.00", "Marseille,Clothing,12.00",
"Paris,Food,8.00", "Paris,Clothing,15.00",
"Marseille,Food,20.00", "Lyon,Book,10.00"]))

In [ ]:
schema = StructType([
                     StructField("city", StringType(), nullable=True),
                     StructField("type", StringType(), nullable=True),
                     StructField("price", FloatType(), nullable=True)
])


In [ ]:
data_list = list(map(lambda r: (r[0], r[1], float(r[2])),
                     map(lambda x: x.split(","),["Paris,Food,19.00", "Marseille,Clothing,12.00",
                                                "Paris,Food,8.00", "Paris,Clothing,15.00",
                                                "Marseille,Food,20.00", "Lyon,Book,10.00"])))

df = spark.createDataFrame(data_list, schema)

In [ ]:
df.show()

+---------+--------+-----+
|     city|    type|price|
+---------+--------+-----+
|    Paris|    Food| 19.0|
|Marseille|Clothing| 12.0|
|    Paris|    Food|  8.0|
|    Paris|Clothing| 15.0|
|Marseille|    Food| 20.0|
|     Lyon|    Book| 10.0|
+---------+--------+-----+



In [ ]:
df.filter(df.city == "Paris").show()

+-----+--------+-----+
| city|    type|price|
+-----+--------+-----+
|Paris|    Food| 19.0|
|Paris|    Food|  8.0|
|Paris|Clothing| 15.0|
+-----+--------+-----+



In [ ]:
df.filter((df.city == "Paris") & (df.type == "Food")).show()

+-----+----+-----+
| city|type|price|
+-----+----+-----+
|Paris|Food| 19.0|
|Paris|Food|  8.0|
+-----+----+-----+



In [ ]:
df.orderBy(df.city).orderBy(df.type).show()

+---------+--------+-----+
|     city|    type|price|
+---------+--------+-----+
|     Lyon|    Book| 10.0|
|Marseille|Clothing| 12.0|
|    Paris|Clothing| 15.0|
|    Paris|    Food| 19.0|
|Marseille|    Food| 20.0|
|    Paris|    Food|  8.0|
+---------+--------+-----+



In [ ]:
df.withColumn("divide",df.price/2).show()

+---------+--------+-----+------+
|     city|    type|price|divide|
+---------+--------+-----+------+
|    Paris|    Food| 19.0|   9.5|
|Marseille|Clothing| 12.0|   6.0|
|    Paris|    Food|  8.0|   4.0|
|    Paris|Clothing| 15.0|   7.5|
|Marseille|    Food| 20.0|  10.0|
|     Lyon|    Book| 10.0|   5.0|
+---------+--------+-----+------+



In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import sum as _sum
df.groupBy("city").agg(_sum("price")).show()

+---------+----------+
|     city|sum(price)|
+---------+----------+
|Marseille|      32.0|
|    Paris|      42.0|
|     Lyon|      10.0|
+---------+----------+



In [ ]:
# Simulate heritage claim data with messy bureau codes
data = [
    ("ilu-001", "John Smith", "GBP", 1500.00),
    ("lirma-042", "Jane Doe", "USD", 2300.50),
    ("ilu-003", "Bob Wilson", "EUR", 800.00),
    ("lloyds-017", "Alice Brown", "GBP", 4200.00),
]

In [ ]:
columns = ["bureau_code", "underwriter", "currency", "amount"]

In [ ]:
df = spark.createDataFrame(data, columns)

In [ ]:
df.show()

+-----------+-----------+--------+------+
|bureau_code|underwriter|currency|amount|
+-----------+-----------+--------+------+
|    ilu-001| John Smith|     GBP|1500.0|
|  lirma-042|   Jane Doe|     USD|2300.5|
|    ilu-003| Bob Wilson|     EUR| 800.0|
| lloyds-017|Alice Brown|     GBP|4200.0|
+-----------+-----------+--------+------+



In [ ]:
# Write a UDF that extracts the bureau name (part before the hyphen)
# "ilu-001" → "ILU"
# "lirma-042" → "LIRMA"
# "lloyds-017" → "LLOYDS"
